# Batched Groq LLM Evaluation Runner

This notebook reads `data/processed/evaluation.json`, loads each `source_file`, sends batched PRs to Groq `gpt-oss-20b`, and appends raw model outputs to `outputs/llm_raw_responses.jsonl`.

In [1]:
import os, json, time
from dotenv import load_dotenv
import pandas as pd
from pathlib import Path
from groq import Groq

ROOT = Path("..").resolve()
EVAL_PATH = ROOT / 'data' / 'processed' / 'evaluation.json'
RAW_OUT = ROOT / 'outputs' / 'llm_raw_responses.txt'
ZERO_RESPONSE_PR = ROOT / 'outputs' / 'zero_response_PRs.txt'
PARSED_OUT = ROOT / 'outputs' / 'llm_reviews.json'
RAW_OUT.parent.mkdir(parents=True, exist_ok=True)
ZERO_RESPONSE_PR.parent.mkdir(parents=True, exist_ok=True)
PARSED_OUT.parent.mkdir(parents=True, exist_ok=True)

MODEL = 'openai/gpt-oss-20b'
BATCH_SIZE = 1
MAX_CHARS_PER_FILE = 8000

# evaluation.json range controls (inclusive, 0-based)
STARTING = 0
ENDING = 29

# Token controls to stay below Groq on_demand TPM limit
MODEL_CONTEXT_LIMIT = 8192  # conservative safe value
MAX_OUTPUT_TOKENS = 800

INPUT_TOKEN_BUDGET = int(MODEL_CONTEXT_LIMIT * 0.75) - MAX_OUTPUT_TOKENS
CHARS_PER_TOKEN = 2.5  # safer for code

load_dotenv()  # Load .env file if present

GROQ_API_KEY = os.environ['GROQ_API_KEY']
GROQ_API_URL = os.environ.get('GROQ_API_URL', 'https://api.groq.com')
client = Groq(api_key=GROQ_API_KEY, base_url=GROQ_API_URL)

In [2]:
def load_evaluation(path: Path):
    data = json.loads(path.read_text(encoding='utf-8'))
    return [r for r in data if isinstance(r, dict) and r.get('source_file')]

def read_source_text(source_file: str):
    p = ROOT / source_file
    if not p.exists():
        p = ROOT / 'data' / 'processed' / source_file
    text = p.read_text(encoding='utf-8')
    if len(text) <= MAX_CHARS_PER_FILE:
        return text
    half = MAX_CHARS_PER_FILE // 2
    return text[:half] + '\n\n...TRUNCATED...\n\n' + text[-half:]

def est_tokens(text: str):
    return max(1, len(text) // CHARS_PER_TOKEN)

records = load_evaluation(EVAL_PATH)
records = records[STARTING:ENDING + 1]
print('selected_records:', len(records), 'range:', STARTING, 'to', ENDING, '(inclusive)')

selected_records: 30 range: 0 to 29 (inclusive)


In [3]:
rows = []
row_idx = 1
for entry in records:
    pr_id = entry['id']
    for review in entry.get('ground_truth_reviews', []):
        rows.append({
            'id': f"row_{row_idx}" ,
            'PR': pr_id,
            'line_number': review.get('line_number'),
            'violation': review.get('violation_category'),
            'review_comment': review.get('review_comment')
        })
        row_idx += 1

ground_truth_df = pd.DataFrame(rows, columns=['id', 'PR', 'line_number', 'violation', 'review_comment'])
print('ground_truth_df shape:', ground_truth_df.shape)
ground_truth_df.head(10)

ground_truth_df shape: (199, 5)


,id,PR,line_number,violation,review_comment
0,row_1,synthetic-django_PR_21,9,unused_import,Unused import: os
1,row_2,synthetic-django_PR_21,10,unused_import,Unused import: sys
2,row_3,synthetic-django_PR_21,11,unused_import,Unused import: re
3,row_4,synthetic-django_PR_22,43,naming_convention,camelCase function name: cleanTitle
4,row_5,synthetic-django_PR_22,76,naming_convention,camelCase function name: cleanName
5,row_6,synthetic-django_PR_23,11,unused_import,Unused import: os
6,row_7,synthetic-django_PR_23,12,unused_import,Unused import: sys
7,row_8,synthetic-django_PR_23,13,unused_import,Unused import: re
8,row_9,synthetic-django_PR_23,24,indentation,Non-4-space indent (2 spaces)
9,row_10,synthetic-django_PR_23,34,indentation,Non-4-space indent (6 spaces)


In [4]:
def build_prompt(batch):
    header = """TASK: Detect STRICT PEP 8-style violations in Python code.

PROCESSING RULES:
- Process EACH PR independently.
- After finishing ONE PR, immediately output its JSON object.
- Do NOT wait for other PRs.
- Do NOT analyze across PRs.

VIOLATION TYPES (ONLY these):

1) naming_convention:
   - Functions, variables, parameters → must be snake_case
   - Classes → must be PascalCase (CapWords)
   - Constants → must be UPPER_CASE
   - Flag:
       * camelCase names in functions/variables/params
       * PascalCase used for functions/variables
       * snake_case used for class names

2) indentation:
   - Indentation must be exactly 4 spaces per level
   - Flag:
       * tabs used
       * non-multiple of 4 spaces
       * inconsistent indentation within block

3) unused_import:
   - Imported module/symbol not referenced anywhere in file

4) mutable_default:
   - Function parameters using [] or {} as default values

5) documentation_formatting:
   - Docstring indentation inconsistent with block level
   - Misaligned triple quotes
   - Broken or inconsistent docstring structure

CONSTRAINTS:
- Max 5 findings per PR
- Detect ONLY clear, high-confidence violations
- Do NOT guess or infer
- Be precise: exact line_number

REVIEW COMMENT STYLE:
- Short, technical, actionable
- Mention issue + direct fix
- No explanations

OUTPUT (STRICT):
Return ONLY a JSON array:
[
  {
    "PR_ID": "string",
    "llm_reviews": [
      {
        "line_number": int,
        "violation_category": "string",
        "review_comment": "string"
      }
    ]
  }
]

CRITICAL:
- No markdown
- No explanations
- No reasoning text
- Output immediately per PR
"""

    blocks = [header]

    for item in batch:
        code = item['source_code']

        blocks.append(
f"""PR:
ID: {item['id']}
CODE:
{code}
END
"""
        )

    return '\n'.join(blocks)

In [5]:
def call_groq(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {'role': 'user', 'content': prompt}
        ]
    )
    return response

In [6]:
# Single-sample Groq run: pick the first entry and print response (no file writes)
sample = records[0]
entry = {
    'id': sample['id'],
    'repo': sample.get('repo'),
    'source_file': sample['source_file'],
    'source_code': read_source_text(sample['source_file'])
}
prompt = build_prompt([entry])
print('=== PROMPT (truncated 2000 chars) ===')
print(prompt[:2000])
print('\n=== CALLING GROQ ===')
resp = call_groq(prompt)
content = resp.choices[0].message.content
print('\n=== RESPONSE ===')
print(content)

=== PROMPT (truncated 2000 chars) ===
TASK: Detect STRICT PEP 8-style violations in Python code.

PROCESSING RULES:
- Process EACH PR independently.
- After finishing ONE PR, immediately output its JSON object.
- Do NOT wait for other PRs.
- Do NOT analyze across PRs.

VIOLATION TYPES (ONLY these):

1) naming_convention:
   - Functions, variables, parameters → must be snake_case
   - Classes → must be PascalCase (CapWords)
   - Constants → must be UPPER_CASE
   - Flag:
       * camelCase names in functions/variables/params
       * PascalCase used for functions/variables
       * snake_case used for class names

2) indentation:
   - Indentation must be exactly 4 spaces per level
   - Flag:
       * tabs used
       * non-multiple of 4 spaces
       * inconsistent indentation within block

3) unused_import:
   - Imported module/symbol not referenced anywhere in file

4) mutable_default:
   - Function parameters using [] or {} as default values

5) documentation_formatting:
   - Docstrin

In [7]:
import time

batch_size = BATCH_SIZE
dry_run = False

prepared = []
for r in records:
    prepared.append({
        'id': r['id'],
        'repo': r['repo'],
        'source_file': r['source_file'],
        'source_code': read_source_text(r['source_file'])
    })

id_to_repo = {x['id']: x['repo'] for x in prepared}

zero_response_prs = set()

i = 0
batch_no = 0

while i < len(prepared):
    time.sleep(2)  # brief pause between batches

    batch = []
    while i < len(prepared) and len(batch) < BATCH_SIZE:
        candidate = prepared[i]
        batch.append(candidate)
        i += 1

    batch_no += 1
    prompt = build_prompt(batch)

    resp = call_groq(prompt)
    choice0 = resp.choices[0]
    message0 = choice0.message
    content = message0.content

    # Print diagnostics
    print(f'\n=== BATCH {batch_no} RESPONSE DIAGNOSTICS ===')
    pr_ids = [p["id"] for p in batch]
    print(f'PR ids: {pr_ids}')
    print(f'content_type: {type(content).__name__}')

    is_empty = False

    if isinstance(content, str):
        content_stripped = content.strip()
        print(f'content_len: {len(content)}')
        print(f'is_empty_after_strip: {len(content_stripped) == 0}')
        print('response_preview:')
        print(content[:1200])

        if len(content_stripped) == 0:
            is_empty = True
    else:
        print('response_preview_non_str:')
        print(content)
        is_empty = True  # treat non-str as failure

    if is_empty:
        print(f'Empty response detected for batch {batch_no}')
        zero_response_prs.update(pr_ids)

    content_to_write = content if isinstance(content, str) else str(content)

    # Append raw response
    with open(RAW_OUT, 'a', encoding='utf-8') as rf:
        rf.write(f'Batch {batch_no} - PR ids: {pr_ids}\n')
        rf.write(f'repos: {[id_to_repo[p_id] for p_id in pr_ids]}\n')
        rf.write(content_to_write)
        rf.write('\n\n\n')

    print(f"Batch {batch_no}: {len(batch)} PRs, est_input_tokens={est_tokens(prompt)}")


if zero_response_prs:
    print(f"\nWriting {len(zero_response_prs)} zero-response PRs to file...")

    with open(ZERO_RESPONSE_PR, 'w', encoding='utf-8') as f:
        for pr_id in sorted(zero_response_prs):
            f.write(f"{pr_id}\n")

    print("Done.")
else:
    print("\nNo zero-response PRs detected.")


=== BATCH 1 RESPONSE DIAGNOSTICS ===
PR ids: ['synthetic-django_PR_21']
content_type: str
content_len: 513
is_empty_after_strip: False
response_preview:
[
  {
    "PR_ID": "synthetic-django_PR_21",
    "llm_reviews": [
      {
        "line_number": 8,
        "violation_category": "unused_import",
        "review_comment": "Remove unused import 'os'."
      },
      {
        "line_number": 9,
        "violation_category": "unused_import",
        "review_comment": "Remove unused import 'sys'."
      },
      {
        "line_number": 10,
        "violation_category": "unused_import",
        "review_comment": "Remove unused import 're'."
      }
    ]
  }
]
Batch 1: 1 PRs, est_input_tokens=1766.0

=== BATCH 2 RESPONSE DIAGNOSTICS ===
PR ids: ['synthetic-django_PR_22']
content_type: str
content_len: 454
is_empty_after_strip: False
response_preview:
[
  {
    "PR_ID": "synthetic-django_PR_22",
    "llm_reviews": [
      {
        "line_number": 42,
        "violation_category": "naming

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01jc5cwt9jfpwrgw214pnvga48` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198099, Requested 3120. Please try again in 8m46.607999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
ground_truth_df[ground_truth_df['PR'] == "synthetic-django_PR_21"]

## Usage

1. Install library: `pip install groq`
2. Export key: `GROQ_API_KEY`
3. In the first code cell, set `STARTING` and `ENDING` (inclusive, 0-based index in `evaluation.json`).
4. Tune token controls in the first code cell: `MAX_OUTPUT_TOKENS` and `INPUT_TOKEN_BUDGET`.
5. Keep `dry_run=True` for prompt inspection, then set `dry_run=False` for real execution.
6. Raw outputs append to `outputs/llm_raw_responses.txt` with PR metadata headers.
7. Parsed outputs are incrementally written to `outputs/llm_reviews.json` after each parsed batch.